In [ ]:
import pandas as pd

arquivo_original = "dados/si-bol-2021.csv"
arquivo_novo_utf8 = "dados/acidentes_transito_utf8.csv"

print("Iniciando a leitura do arquivo original...")

#Lê o arquivo com a codificação do governo e o separador em ponto e vírgula
df = pd.read_csv(arquivo_original, encoding="latin1", sep=";")

#Salva um NOVO arquivo, forçando o padrão universal UTF-8 e separador por vírgula
#O index=False evita que o Pandas crie uma coluna extra de numeração
df.to_csv(arquivo_novo_utf8, encoding="utf-8", sep=",", index=False)

print(f"Sucesso! O arquivo {arquivo_novo_utf8} foi criado e está pronto para uso.")

Iniciando a leitura do arquivo original...
Sucesso! O arquivo acidentes_transito_utf8.csv foi criado e está pronto para uso.


In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe # Importação necessária na nova versão

context = gx.get_context()
batch = context.data_sources.pandas_default.read_csv(
    "dados/acidentes_transito_utf8.csv",
    sep=","
)

#expectativa_1 = gxe.ExpectColumnToExist(column="NUMERO_BOLETIM")
expectativa_1 = gxe.ExpectColumnToExist(column=" NUMERO_BOLETIM")
resultado_1 = batch.validate(expectativa_1)

expectativa_2 = gxe.ExpectColumnValuesToNotBeNull(column=" NUMERO_BOLETIM")
resultado_2 = batch.validate(expectativa_2)

print("--- RESULTADOS DOS TESTES ---")
print(f"1. A coluna 'NUMERO_BOLETIM' existe? {resultado_1.success}")
print(f"2. A coluna 'NUMERO_BOLETIM' está livre de valores nulos? {resultado_2.success}")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

--- RESULTADOS DOS TESTES ---
1. A coluna 'NUMERO_BOLETIM' existe? True
2. A coluna 'NUMERO_BOLETIM' está livre de valores nulos? True


Executando o código acima, a validação falhou indicando que a coluna não existia. Rodando o código abaixo, verifiquei que todas as colunas estavam com um espaço oculto antes do nome, fazendo com que não fossem reconhecidas corretamente.

In [ ]:
import pandas as pd

# Lê o arquivo
df = pd.read_csv("dados/acidentes_transito_utf8.csv", sep=",")

# Imprime a lista EXATA das colunas
print("Lista exata de colunas:")
print(df.columns.tolist())

Lista exata de colunas:
[' NUMERO_BOLETIM', ' DATA HORA_BOLETIM', ' DATA_INCLUSAO', ' TIPO_ACIDENTE', ' DESC_TIPO_ACIDENTE', ' COD_TEMPO', ' DESC_TEMPO', ' COD_PAVIMENTO', ' PAVIMENTO', ' COD_REGIONAL', ' DESC_REGIONAL', ' ORIGEM_BOLETIM', ' LOCAL_SINALIZADO', ' VELOCIDADE_PERMITIDA', ' COORDENADA_X', ' COORDENADA_Y', ' HORA_INFORMADA', ' INDICADOR_FATALIDADE', ' VALOR_UPS', ' DESCRIÇÃO_UPS', ' DATA_ALTERACAO_SMSA', ' VALOR_UPS_ANTIGA', ' DESCRIÇÃO_UPS_ANTIGA']


Para resolver o problema dos nomes das colunas, resolvi fazer um tratamento de dados removendo os espaços e colocando os nomes em letras maiúsculas.

In [ ]:
import pandas as pd

df = pd.read_csv("dados/acidentes_transito_utf8.csv", sep=",")
df.columns = df.columns.str.strip()
df.columns = df.columns.str.upper()
df.to_csv("dados/acidentes_transito_limpo.csv", sep=",", index=False)

print("Limpeza concluída! Veja os nomes das colunas agora:")
print(df.columns.tolist())

Limpeza concluída! Veja os nomes das colunas agora:
['NUMERO_BOLETIM', 'DATA HORA_BOLETIM', 'DATA_INCLUSAO', 'TIPO_ACIDENTE', 'DESC_TIPO_ACIDENTE', 'COD_TEMPO', 'DESC_TEMPO', 'COD_PAVIMENTO', 'PAVIMENTO', 'COD_REGIONAL', 'DESC_REGIONAL', 'ORIGEM_BOLETIM', 'LOCAL_SINALIZADO', 'VELOCIDADE_PERMITIDA', 'COORDENADA_X', 'COORDENADA_Y', 'HORA_INFORMADA', 'INDICADOR_FATALIDADE', 'VALOR_UPS', 'DESCRIÇÃO_UPS', 'DATA_ALTERACAO_SMSA', 'VALOR_UPS_ANTIGA', 'DESCRIÇÃO_UPS_ANTIGA']


Validação e Garantia de Qualidade dos Dados
Antes de iniciarmos a exploração visual, é fundamental garantir que a nossa base de dados é confiável. Para isso, utilizei a biblioteca Great Expectations para aplicar 5 regras de negócio estritas:

1 Integridade da Chave: O NUMERO_BOLETIM não pode ser nulo.

2 Prevenção de Duplicatas: Cada NUMERO_BOLETIM deve ser único para não inflar as estatísticas.

3 Padronização Categórica: A coluna INDICADOR_FATALIDADE só pode conter os valores exatos "SIM" ou "NÃO".

4 Limites Lógicos: A VELOCIDADE_PERMITIDA na via deve estar entre 0 e 120 km/h.

5 Consistência de Data: A coluna de data e hora deve seguir rigorosamente o padrão brasileiro (DD/MM/AAAA HH:MM).

Resultado: Todos os testes retornaram True, atestando a qualidade da base para a etapa de visualização.

In [1]:
import great_expectations as gx
import great_expectations.expectations as gxe

# 1. Inicia o contexto e carrega o arquivo limpo
context = gx.get_context()
batch = context.data_sources.pandas_default.read_csv("dados/acidentes_transito_limpo.csv", sep=",")

# --- CRIANDO AS REGRAS DE NEGÓCIO ---

# Regra 1 e 2: O Boletim não pode ser nulo e não pode ser repetido
regra_boletim_nulo = gxe.ExpectColumnValuesToNotBeNull(column="NUMERO_BOLETIM")
regra_boletim_unico = gxe.ExpectColumnValuesToBeUnique(column="NUMERO_BOLETIM")

# Regra 3: A fatalidade só pode estar escrita como SIM ou NÃO
regra_fatalidade = gxe.ExpectColumnValuesToBeInSet(
    column="INDICADOR_FATALIDADE", 
    value_set=["SIM", "NÃO"]
)

# Regra 4: A velocidade não pode ser menor que 0 nem maior que 120
regra_velocidade = gxe.ExpectColumnValuesToBeBetween(
    column="VELOCIDADE_PERMITIDA", 
    min_value=0, 
    max_value=120
)

# Regra 5: A data deve seguir o padrão: DD/MM/AAAA HH:MM
regra_formato_data = gxe.ExpectColumnValuesToMatchRegex(
    column="DATA HORA_BOLETIM", 
    regex=r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$"
)

# --- EXECUTANDO OS TESTES ---
print("--- RESULTADOS DA QUALIDADE DOS DADOS ---")
print(f"1. Boletim não é nulo? {batch.validate(regra_boletim_nulo).success}")
print(f"2. Boletins são únicos? {batch.validate(regra_boletim_unico).success}")
print(f"3. Fatalidade apenas SIM ou NÃO? {batch.validate(regra_fatalidade).success}")
print(f"4. Velocidade entre 0 e 120? {batch.validate(regra_velocidade).success}")
print(f"5. Datas estão no formato correto? {batch.validate(regra_formato_data).success}")

--- RESULTADOS DA QUALIDADE DOS DADOS ---


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

1. Boletim não é nulo? True


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2. Boletins são únicos? True


...

3. Fatalidade apenas SIM ou NÃO? True


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

4. Velocidade entre 0 e 120? True


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

5. Datas estão no formato correto? True


Com o objetivo de preparar o ambiente para consultas SQL e criar visualizações mais aprofundadas, adicionei duas novas fontes de dados referentes à mortalidade geral (dados do DataSUS) e à população (dados do IBGE) na mesma região dos acidentes.

In [ ]:
import pandas as pd
import io

# 1. Os dados de Belo Horizonte
dados_bh = """Ano do Óbito;Janeiro;Fevereiro;Marco;Abril;Maio;Junho;Julho;Agosto;Setembro;Outubro;Novembro;Dezembro;Total
2021;1874;1544;2551;2553;2072;1845;1841;1587;1445;1445;1364;1486;21607
Total;1874;1544;2551;2553;2072;1845;1841;1587;1445;1445;1364;1486;21607"""

# 2. Ler o texto transformando num DataFrame do Pandas
df_mortalidade = pd.read_csv(io.StringIO(dados_bh), sep=";")

# 3. Limpeza: Remover a linha "Total" e a coluna "Total" para não sujar a análise
df_mortalidade = df_mortalidade[df_mortalidade["Ano do Óbito"] != "Total"] 
df_mortalidade = df_mortalidade.drop(columns=["Total"]) 

# 4. Transforma as 12 colunas de meses em apenas 1 coluna com 12 linhas
df_obitos_bh = df_mortalidade.melt(
    id_vars=["Ano do Óbito"], # Fixa a coluna do ano
    var_name="Mes",           # Nome da nova coluna que receberá os meses
    value_name="Total_Obitos_Gerais" # Nome da coluna que receberá os valores
)

# 5. Salva a nova tabela no computador, já limpa e pronta para o SQL!
df_obitos_bh.to_csv("dados/obitos_gerais_bh_2021.csv", index=False)

# Exibe o resultado final
print("Tabela transformada com sucesso! Veja como ficou:")
print(df_obitos_bh)

Tabela transformada com sucesso! Veja como ficou:
   Ano do Óbito        Mes  Total_Obitos_Gerais
0          2021    Janeiro                 1874
1          2021  Fevereiro                 1544
2          2021      Marco                 2551
3          2021      Abril                 2553
4          2021       Maio                 2072
5          2021      Junho                 1845
6          2021      Julho                 1841
7          2021     Agosto                 1587
8          2021   Setembro                 1445
9          2021    Outubro                 1445
10         2021   Novembro                 1364
11         2021   Dezembro                 1486


In [ ]:
import pandas as pd

arquivo_excel = "dados/populacao.xlsx" 
arquivo_csv_utf8 = "dados/populacao_bh_utf8.csv"

print("Lendo o arquivo Excel...")

# 2. O Pandas lê a planilha original, preservando todos os acentos corretamente
df_excel = pd.read_excel(arquivo_excel)

# 3. Exporta para CSV forçando a codificação UTF-8 e o separador por vírgula
df_excel.to_csv(arquivo_csv_utf8, encoding="utf-8", index=False, sep=",")

print(f"Sucesso! O arquivo '{arquivo_csv_utf8}' foi criado e está pronto para uso.")

Lendo o arquivo Excel...
Sucesso! O arquivo 'populacao_bh_utf8.csv' foi criado e está pronto para uso.


In [2]:
import pandas as pd

# 1. Lê o arquivo intacto
df_ibge = pd.read_csv("dados/populacao_bh_utf8.csv")

# 2. Renomeia as colunas
df_ibge.columns = ["Indicador", "Valor"]

# 3. Encontra a linha da população
linha_populacao = df_ibge[df_ibge["Indicador"].str.contains("População no último censo", na=False)]

# 4. Pega o texto e limpa para virar número
texto_bruto = linha_populacao["Valor"].values[0]
texto_limpo = texto_bruto.replace("pessoas", "").replace("[2022]", "").replace(".", "").strip()

# 5. Salva a população como um número de verdade em uma variável do Python
populacao_bh = int(texto_limpo)

print(f"Dados extraídos com sucesso! A população de Belo Horizonte é: {populacao_bh} habitantes.")

Dados extraídos com sucesso! A população de Belo Horizonte é: 2315560 habitantes.
